# 01 — LedgerRoute baseline and reference window

**LedgerRoute** is a synthetic expense-routing classifier: each row is a corporate card transaction with tabular features, and the label is whether finance must manually review it.

Monitoring starts with a **reference window**—the period you treat as representative of training. We fit on days 0–29 of a stable stream and evaluate on days 30–39 before running drift scenarios.

Narrative context: [The living model](https://arraxis.com/living-model/) on Arraxis.


In [ ]:
# From repo root: pip install -e ".[dev]"
%matplotlib inline

import pandas as pd

from drift_lab import LEDGER_ROUTE_FEATURES, StreamConfig, generate_stream
from drift_lab.analysis import attach_predictions, outcome_summary
from drift_lab.streams import train_reference_model

cfg = StreamConfig(n_days=40, samples_per_day=200, seed=42)
stable = generate_stream("stable", cfg)
print(f"{len(stable):,} rows, days 0–{stable['day'].max()}")
stable.head()


## Feature schema

| Column | Meaning |
|--------|---------|
| `channel_online` | 1 if submitted via online checkout |
| `log_amount` | log1p(expense amount) |
| `mcc_bucket` | merchant category bucket (0–7) |
| `foreign_flag` | cross-border indicator |
| `weekend` | transaction on Sat/Sun |
| `day` | synthetic day index |
| `label` | 1 = needs manual review |

Labels are drawn from a logistic model on the feature vector (see `drift_lab.streams`).


In [ ]:
stable[LEDGER_ROUTE_FEATURES + ['label']].describe().round(3).T


## Fit on days 0–29, evaluate on 30–39


In [ ]:
train = stable[stable["day"] < 30]
holdout = stable[stable["day"] >= 30]

model = train_reference_model(train)
train_scored = attach_predictions(model, train)
holdout_scored = attach_predictions(model, holdout)

for name, part in [("Train", train_scored), ("Holdout", holdout_scored)]:
    m = outcome_summary(part["label"], part["pred_label"], part["prob_review"])
    print(name, {k: round(v, 4) for k, v in m.items()})


## Label rate by day (stable stream)


In [ ]:
ax = stable.groupby("day")["label"].mean().plot(figsize=(8, 3), title="Review rate by day (stable)")
ax.set_ylabel("P(manual review)")
ax.figure.tight_layout()


## Takeaway

1. Freeze **which days** define your reference window and persist them in config.
2. Store **model version + feature schema** with every prediction row before drift tests.
3. Hold-out on adjacent stable days is a sanity check—not a substitute for production monitoring.
